In [1]:
# src/utils

import sys
sys.path.append("/home/jovyan/work")

from src.utils.spark_session import get_spark_session
spark = get_spark_session("generate-sample-data")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [3]:
# Reference/ congig tables

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType
from datetime import datetime

now = datetime.now()

# --- branch ---
df_branch = spark.createDataFrame([
    Row(branch_id=1, name="Lagos - Head Office", address="Ikeja, Lagos",
        is_active=True, created_at=now, updated_at=now),
])
df_branch.write.format("delta").mode("overwrite").saveAsTable("silver.branch")

# --- data_source ---
df_data_source = spark.createDataFrame([
    Row(source_id=1, source_code="GOOGLE_FORM", description="Google Forms submission"),
    Row(source_id=2, source_code="WHATSAPP_GROUP_AGENT", description="WhatsApp group message, LLM-extracted"),
    Row(source_id=3, source_code="RECEIPT_AGENT", description="WhatsApp receipt photo, OCR-extracted"),
    Row(source_id=4, source_code="MANUAL", description="Manual entry / correction"),
])
df_data_source.write.format("delta").mode("overwrite").saveAsTable("silver.data_source")

# --- auction_house ---
df_auction_house = spark.createDataFrame([
    Row(auction_house_id=1, name="Copart", country="USA", created_at=now),
    Row(auction_house_id=2, name="IAAI", country="USA", created_at=now),
    Row(auction_house_id=3, name="Manheim", country="USA", created_at=now),
])
df_auction_house.write.format("delta").mode("overwrite").saveAsTable("silver.auction_house")

# --- vendor ---
df_vendor = spark.createDataFrame([
    Row(vendor_id=1, name="AutoParts Lagos Ltd", contact_phone="+2348012345001", contact_email="sales@autopartslagos.ng", created_at=now),
    Row(vendor_id=2, name="Naija Spares Hub", contact_phone="+2348012345002", contact_email="info@naijaspares.ng", created_at=now),
    Row(vendor_id=3, name="Trust Motor Spares", contact_phone="+2348012345003", contact_email=None, created_at=now),
])
df_vendor.write.format("delta").mode("overwrite").saveAsTable("silver.vendor")

# --- part ---
df_part = spark.createDataFrame([
    Row(part_id=1, name="Front Bumper", category="Body", created_at=now),
    Row(part_id=2, name="Headlight Assembly", category="Body", created_at=now),
    Row(part_id=3, name="Side Mirror", category="Body", created_at=now),
    Row(part_id=4, name="Windshield", category="Body", created_at=now),
    Row(part_id=5, name="Brake Pads", category="Mechanical", created_at=now),
    Row(part_id=6, name="Alternator", category="Electrical", created_at=now),
    Row(part_id=7, name="Radiator", category="Mechanical", created_at=now),
    Row(part_id=8, name="Seat Cover", category="Interior", created_at=now),
])
df_part.write.format("delta").mode("overwrite").saveAsTable("silver.part")

# --- staff ---
df_staff = spark.createDataFrame([
    Row(staff_id=1, branch_id=1, name="Tunde Bakare", phone_whatsapp="+2348030000001", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=2, branch_id=1, name="Chinedu Okafor", phone_whatsapp="+2348030000002", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=3, branch_id=1, name="Musa Ibrahim", phone_whatsapp="+2348030000003", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=4, branch_id=1, name="Ifeoma Nwosu", phone_whatsapp="+2348030000004", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=5, branch_id=1, name="Segun Adeyemi", phone_whatsapp="+2348030000005", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=6, branch_id=1, name="Amaka Eze", phone_whatsapp="+2348030000006", is_active=True, created_at=now, updated_at=now),
])
df_staff.write.format("delta").mode("overwrite").saveAsTable("silver.staff")


# --- staff_role (who does what — matches "roles not fixed headcount" from Phase 1) ---
staff_role_schema = StructType([
    StructField("staff_id", LongType(), False),
    StructField("role_code", StringType(), False),
    StructField("valid_from", DateType(), False),
    StructField("valid_to", DateType(), True),
])

df_staff_role = spark.createDataFrame([
    Row(staff_id=1, role_code="OWNER", valid_from=now.date(), valid_to=None),
    Row(staff_id=1, role_code="FINANCE", valid_from=now.date(), valid_to=None),
    Row(staff_id=2, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=3, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="INSPECTOR", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=5, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=6, role_code="SALES", valid_from=now.date(), valid_to=None),
], schema=staff_role_schema)
df_staff_role.write.format("delta").mode("overwrite").saveAsTable("silver.staff_role")


# --- inspection_checklist_item ---
df_checklist = spark.createDataFrame([
    Row(item_id=1, category="ENGINE", label="Engine starts and runs smoothly", is_active=True, sort_order=1),
    Row(item_id=2, category="ENGINE", label="No visible oil/coolant leaks", is_active=True, sort_order=2),
    Row(item_id=3, category="BODY", label="No major dents or rust", is_active=True, sort_order=3),
    Row(item_id=4, category="BODY", label="Panel alignment is correct", is_active=True, sort_order=4),
    Row(item_id=5, category="ELECTRICAL", label="All lights functional", is_active=True, sort_order=5),
    Row(item_id=6, category="ELECTRICAL", label="AC/heating works", is_active=True, sort_order=6),
    Row(item_id=7, category="INTERIOR", label="Seats and upholstery in good condition", is_active=True, sort_order=7),
    Row(item_id=8, category="TIRES", label="Tire tread depth acceptable", is_active=True, sort_order=8),
])
df_checklist.write.format("delta").mode("overwrite").saveAsTable("silver.inspection_checklist_item")

for t in ["branch", "data_source", "auction_house", "vendor", "part", "staff", "staff_role", "inspection_checklist_item"]:
    print(f"silver.{t}: {spark.table(f'silver.{t}').count()} rows")

silver.branch: 1 rows
silver.data_source: 4 rows
silver.auction_house: 3 rows
silver.vendor: 3 rows
silver.part: 8 rows
silver.staff: 6 rows
silver.staff_role: 8 rows
silver.inspection_checklist_item: 8 rows


In [4]:
from faker import Faker
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, IntegerType,
    DateType, TimestampType, BooleanType, DecimalType
)
from decimal import Decimal
from datetime import timedelta
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

MAKE_MODELS = [
    ("Toyota", "Camry"), ("Toyota", "Corolla"), ("Toyota", "Highlander"), ("Toyota", "RAV4"),
    ("Honda", "Accord"), ("Honda", "CR-V"), ("Honda", "Civic"),
    ("Lexus", "RX350"), ("Lexus", "ES350"),
    ("Ford", "Edge"), ("Ford", "Explorer"),
    ("Nissan", "Altima"), ("Nissan", "Murano"),
]

N_VEHICLES = 40
today = datetime.now()

vehicle_schema = StructType([
    StructField("vehicle_id", LongType(), False),
    StructField("vin", StringType(), True),
    StructField("auction_lot_no", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("branch_id", LongType(), False),
    StructField("current_stage", StringType(), False),
    StructField("current_status", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

purchase_schema = StructType([
    StructField("purchase_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("auction_house_id", LongType(), True),
    StructField("buyer_staff_id", LongType(), True),
    StructField("price_amount", DecimalType(14, 2), False),
    StructField("price_currency", StringType(), False),
    StructField("purchase_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

vehicle_rows, purchase_rows = [], []

for i in range(1, N_VEHICLES + 1):
    make, model = random.choice(MAKE_MODELS)
    purchase_date = today - timedelta(days=random.randint(1, 180))
    price = Decimal(str(round(random.uniform(1500, 9000), 2)))

    vehicle_rows.append(Row(
        vehicle_id=i,
        vin=fake.bothify(text="?????????????????").upper(),
        auction_lot_no=fake.bothify(text="LOT-#####"),
        make=make, model=model,
        model_year=random.randint(2012, 2020),
        branch_id=1,
        current_stage="PURCHASED",
        current_status="ACTIVE",
        created_at=purchase_date, updated_at=purchase_date,
        is_deleted=False, deleted_at=None,
    ))

    purchase_rows.append(Row(
        purchase_id=i, vehicle_id=i,
        auction_house_id=random.randint(1, 3),
        buyer_staff_id=1,
        price_amount=price,
        price_currency="USD",
        purchase_date=purchase_date.date(),
        status="CONFIRMED",
        source_id=1,
        source_record_id=f"FORM-PUR-{i:04d}",
        created_at=purchase_date, updated_at=purchase_date,
        is_deleted=False, deleted_at=None,
    ))

df_vehicle = spark.createDataFrame(vehicle_rows, schema=vehicle_schema)
df_vehicle.write.format("delta").mode("overwrite").saveAsTable("silver.vehicle")

df_purchase = spark.createDataFrame(purchase_rows, schema=purchase_schema)
df_purchase.write.format("delta").mode("overwrite").saveAsTable("silver.purchase")

print(f"silver.vehicle: {spark.table('silver.vehicle').count()} rows")
print(f"silver.purchase: {spark.table('silver.purchase').count()} rows")
spark.table("silver.vehicle").show(5)

silver.vehicle: 40 rows
silver.purchase: 40 rows
+----------+-----------------+--------------+------+--------+----------+---------+-------------+--------------+--------------------+--------------------+----------+----------+
|vehicle_id|              vin|auction_lot_no|  make|   model|model_year|branch_id|current_stage|current_status|          created_at|          updated_at|is_deleted|deleted_at|
+----------+-----------------+--------------+------+--------+----------+---------+-------------+--------------+--------------------+--------------------+----------+----------+
|        21|NTVNROQGFQDFOBRCA|     LOT-52427|  Ford|Explorer|      2014|        1|    PURCHASED|        ACTIVE|2026-03-22 08:43:...|2026-03-22 08:43:...|     false|      NULL|
|        22|JTBJAHESJICXLJJBI|     LOT-04505|Nissan|  Altima|      2018|        1|    PURCHASED|        ACTIVE|2026-02-24 08:43:...|2026-02-24 08:43:...|     false|      NULL|
|        23|NRPQGWXJANVJPKZZL|     LOT-60256| Honda|   Civic|      2014

In [5]:
purchases = spark.table("silver.purchase").select("vehicle_id", "purchase_date").collect()
today_date = today.date()

shipment_schema = StructType([
    StructField("shipment_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("carrier", StringType(), True),
    StructField("bill_of_lading_no", StringType(), True),
    StructField("etd", DateType(), True),
    StructField("eta", DateType(), True),
    StructField("ata", DateType(), True),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

CARRIERS = ["Maersk Line", "MSC", "CMA CGM", "Hapag-Lloyd", "COSCO Shipping"]

shipment_rows = []
shipment_id = 1

for row in purchases:
    purchase_date = row.purchase_date
    elapsed_days = (today_date - purchase_date).days

    if elapsed_days < 2:
        continue  # too soon to have been booked/shipped yet

    etd = purchase_date + timedelta(days=2)
    shipping_duration = random.randint(20, 45)  # matches the 20-45 day SLA from Phase 1
    eta = etd + timedelta(days=shipping_duration)
    ata = eta if elapsed_days >= (2 + shipping_duration) else None
    status = "ARRIVED" if ata else "IN_TRANSIT"

    shipment_rows.append(Row(
        shipment_id=shipment_id,
        vehicle_id=row.vehicle_id,
        carrier=random.choice(CARRIERS),
        bill_of_lading_no=fake.bothify(text="BL########"),
        etd=etd, eta=eta, ata=ata,
        status=status,
        source_id=1,
        source_record_id=f"FORM-SHIP-{shipment_id:04d}",
        created_at=datetime.combine(etd, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    shipment_id += 1

df_shipment = spark.createDataFrame(shipment_rows, schema=shipment_schema)
df_shipment.write.format("delta").mode("overwrite").saveAsTable("silver.shipment")

print(f"silver.shipment: {spark.table('silver.shipment').count()} rows (out of {len(purchases)} vehicles)")
df_shipment.groupBy("status").count().show()

silver.shipment: 40 rows (out of 40 vehicles)
+----------+-----+
|    status|count|
+----------+-----+
|IN_TRANSIT|    4|
|   ARRIVED|   36|
+----------+-----+



In [6]:
shipments = spark.table("silver.shipment").filter("status = 'ARRIVED'").select("vehicle_id", "ata").collect()

port_arrival_schema = StructType([
    StructField("port_arrival_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("arrival_date", DateType(), False),
    StructField("clearance_status", StringType(), False),
    StructField("cleared_date", DateType(), True),
    StructField("hold_reason", StringType(), True),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

HOLD_REASONS = [
    "Awaiting duty payment confirmation",
    "Missing bill of lading endorsement",
    "Customs valuation dispute",
    "Documentation mismatch - VIN discrepancy",
]

port_arrival_rows = []
pid = 1

for row in shipments:
    arrival_date = row.ata
    elapsed = (today_date - arrival_date).days
    is_delayed = random.random() < 0.15
    normal_duration = random.randint(3, 14)
    extra_delay = random.randint(10, 30) if is_delayed else 0
    total_duration = normal_duration + extra_delay

    if elapsed >= total_duration:
        clearance_status = "CLEARED"
        cleared_date = arrival_date + timedelta(days=total_duration)
        hold_reason = random.choice(HOLD_REASONS) if is_delayed else None
    elif is_delayed:
        clearance_status = "ON_HOLD"
        cleared_date = None
        hold_reason = random.choice(HOLD_REASONS)
    else:
        clearance_status = "PENDING"
        cleared_date = None
        hold_reason = None

    port_arrival_rows.append(Row(
        port_arrival_id=pid,
        vehicle_id=row.vehicle_id,
        arrival_date=arrival_date,
        clearance_status=clearance_status,
        cleared_date=cleared_date,
        hold_reason=hold_reason,
        source_id=1,
        source_record_id=f"FORM-PORT-{pid:04d}",
        created_at=datetime.combine(arrival_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    pid += 1

df_port_arrival = spark.createDataFrame(port_arrival_rows, schema=port_arrival_schema)
df_port_arrival.write.format("delta").mode("overwrite").saveAsTable("silver.port_arrival")

print(f"silver.port_arrival: {spark.table('silver.port_arrival').count()} rows")
df_port_arrival.groupBy("clearance_status").count().show()

silver.port_arrival: 36 rows
+----------------+-----+
|clearance_status|count|
+----------------+-----+
|         CLEARED|   32|
|         PENDING|    3|
|         ON_HOLD|    1|
+----------------+-----+



In [7]:
cleared = spark.table("silver.port_arrival").filter("clearance_status = 'CLEARED'").select("vehicle_id", "cleared_date").collect()

pickup_schema = StructType([
    StructField("pickup_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("driver_staff_id", LongType(), True),
    StructField("pickup_date", DateType(), False),
    StructField("odometer", IntegerType(), True),
    StructField("photo_url", StringType(), True),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

office_intake_schema = StructType([
    StructField("intake_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("intake_date", DateType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

DRIVER_STAFF_IDS = [2, 3]  # from staff_role: role_code = 'DRIVER'

pickup_rows, intake_rows = [], []
pid, iid = 1, 1

for row in cleared:
    elapsed = (today_date - row.cleared_date).days
    if elapsed < 1:
        continue

    pickup_date = row.cleared_date + timedelta(days=1)

    pickup_rows.append(Row(
        pickup_id=pid,
        vehicle_id=row.vehicle_id,
        driver_staff_id=random.choice(DRIVER_STAFF_IDS),
        pickup_date=pickup_date,
        odometer=random.randint(35000, 130000),
        photo_url=f"https://placeholder.local/pickup/{row.vehicle_id}.jpg",
        source_id=1,
        source_record_id=f"FORM-PICKUP-{pid:04d}",
        created_at=datetime.combine(pickup_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))

    intake_rows.append(Row(
        intake_id=iid,
        vehicle_id=row.vehicle_id,
        intake_date=pickup_date,
        source_id=1,
        source_record_id=f"FORM-INTAKE-{iid:04d}",
        created_at=datetime.combine(pickup_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    pid += 1
    iid += 1

df_pickup = spark.createDataFrame(pickup_rows, schema=pickup_schema)
df_pickup.write.format("delta").mode("overwrite").saveAsTable("silver.pickup")

df_intake = spark.createDataFrame(intake_rows, schema=office_intake_schema)
df_intake.write.format("delta").mode("overwrite").saveAsTable("silver.office_intake")

print(f"silver.pickup: {spark.table('silver.pickup').count()} rows")
print(f"silver.office_intake: {spark.table('silver.office_intake').count()} rows")

silver.pickup: 32 rows
silver.office_intake: 32 rows
